<a href="https://colab.research.google.com/github/AdelineKwakye/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-16%20%E2%80%94%20Pandas%20Core%20Ops%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Core Ops

**Studio — 2026-09-16 · Fall 2026**  
**Class time:** 45 minutes

---

## Two halves today

**Part 1** is a short drill on the copy trap from Monday, because it is the bug that quietly changes your numbers instead of raising an error. Budget about 15 minutes.

**Part 2** is one deliverable end to end: a per-vendor summary a game-day manager could act on. Three sources, a join that does not behave, and a report at the end.

The Part 2 data has problems planted in it. Finding them is part of the work — a report you cannot defend is worth nothing, however good the code looks.

---

## Part 1 — The copy trap, with the damage visible

Monday you saw `.copy()` on a slide. Here you watch what happens without it, which is the only way it sticks.

Run the next cell for a small frame to experiment on. The real files arrive in Part 2.

In [1]:
import pandas as pd

print('pandas', pd.__version__)

sales = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'item': ['Rain Poncho', 'Cheeseburger', 'Rain Poncho', 'Hot Dog', 'Foam Finger'],
    'qty': [5, 2, 8, 3, 1],
    'price': [6.00, 7.50, 6.00, 4.50, 12.00],
})
sales

pandas 2.2.3


,order_id,item,qty,price
0,101,Rain Poncho,5,6.0
1,102,Cheeseburger,2,7.5
2,103,Rain Poncho,8,6.0
3,104,Hot Dog,3,4.5
4,105,Foam Finger,1,12.0


### Bad result 1 — the edit that goes nowhere

The ponchos went on sale at $4.50. This is how nearly everybody writes it the first time. Run it, and read the two printed prices before you read any warning.

In [2]:
print('before:', sales.loc[sales['item'] == 'Rain Poncho', 'price'].tolist())

sales[sales['item'] == 'Rain Poncho']['price'] = 4.50

print('after: ', sales.loc[sales['item'] == 'Rain Poncho', 'price'].tolist())

before: [6.0, 6.0]
after:  [6.0, 6.0]


/tmp/ipykernel_15723/3377263661.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales[sales['item'] == 'Rain Poncho']['price'] = 4.50


Nothing changed, and nothing crashed.

That line is two operations, not one. `sales[sales['item'] == 'Rain Poncho']` builds a brand-new temporary frame holding the two poncho rows. `['price'] = 4.50` then sets the price **on that temporary**. Nothing was holding a reference to it, so Python discarded it a microsecond later. Your data never saw the change.

This is the expensive kind of bug. The code looks right, it does not raise, and the number you report is the old one.

### Bad result 2 — the edit that lands somewhere you did not mean

Now the version where the slice goes into a variable first.

In [3]:
rain = sales[sales['item'] == 'Rain Poncho']
rain['sale_price'] = 4.50

print(rain)
print()
print("'sale_price' in rain? ", 'sale_price' in rain.columns)
print("'sale_price' in sales?", 'sale_price' in sales.columns)

   order_id         item  qty  price  sale_price
0       101  Rain Poncho    5    6.0         4.5
2       103  Rain Poncho    8    6.0         4.5

'sale_price' in rain?  True
'sale_price' in sales? False


/tmp/ipykernel_15723/2878508865.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rain['sale_price'] = 4.50


This one did something — `rain` has the new column. What it did not do is touch `sales`.

Whether you also get a warning here depends on your pandas version, which is exactly why the printed columns matter more than the warnings:

- On **pandas 2.x** you get `SettingWithCopyWarning`, because older pandas could not promise whether `rain` shared memory with `sales`.
- On **pandas 3.x** you get nothing at all. Copy-on-Write is always on, so `rain` is guaranteed to be its own frame.

The warning was never the lesson. The lesson is that you have to know which frame your assignment lands in, and slicing never hands you the original.

### The fix — two tools, for two different jobs

Decide what you want before you type. There are exactly two answers.

In [4]:
# Tool 1 -- you want a separate frame to work on. Say so with .copy().
rain = sales[sales['item'] == 'Rain Poncho'].copy()
rain['sale_price'] = 4.50
print('rain has sale_price   :', 'sale_price' in rain.columns)
print('sales left alone      :', 'sale_price' not in sales.columns)
print()

# Tool 2 -- you want to change the original. One operation, so it lands.
sales.loc[sales['item'] == 'Rain Poncho', 'price'] = 4.50
print('sales poncho price now:',
      sales.loc[sales['item'] == 'Rain Poncho', 'price'].tolist())

rain has sale_price   : True
sales left alone      : True

sales poncho price now: [4.5, 4.5]


Keep this table until it is reflex:

| What you want | What to write |
|---|---|
| A separate frame you can modify freely | `sub = df[mask].copy()` |
| To change the original in place | `df.loc[mask, 'col'] = value` |
| Nothing, ever | `df[mask]['col'] = value` |

The third row is not a style preference. It does not work.

### Your turn 1 — make the change actually land

The Hot Dog price should be `5.00` in `sales` itself. The broken version is sitting in the cell as a comment; do not use it. Write the version that works, then print the price to prove it.

In [5]:
# Broken -- leave it commented, it silently does nothing:
#     sales[sales['item'] == 'Hot Dog']['price'] = 5.00

# TODO: the version that changes `sales`
sales.loc[sales['item'] == 'Hot Dog', 'price'] = 5.00
# TODO: print the Hot Dog price -- expect [5.0]
print('Hot dog price:',
       sales.loc[sales['item'] == 'Hot Dog', 'price'].tolist())

Hot dog price: [5.0]


### Your turn 2 — a working copy that leaves the original alone

Build `bulk`: only the rows with `qty >= 3`, plus a new `line_total` column equal to `qty * price`. `sales` must come out of this unchanged.

Two things must be true when you are done: `bulk` has a `line_total` column, and `sales` does not.

In [6]:
# TODO: bulk = ...
bulk = sales[sales['qty']>= 3].copy()
# TODO: bulk['line_total'] = ...
bulk['line_total'] = bulk['qty'] * bulk['price']
# Uncomment these to check yourself:
assert 'line_total' in bulk.columns
assert 'line_total' not in sales.columns
print(bulk)

   order_id         item  qty  price  line_total
0       101  Rain Poncho    5    4.5        22.5
2       103  Rain Poncho    8    4.5        36.0
3       104      Hot Dog    3    5.0        15.0


---

## Part 2 — The vendor report

Three sources: order lines, a vendor roster, and a revenue target per zone. Run the next cell to load them.

In [7]:
import pandas as pd
from io import StringIO

orders = pd.read_csv(StringIO('''order_id,vendor_id,item,qty,price
1,V-01,Cheeseburger,2,7.50
2,V-10,Foam Finger,1,12.00
3,V-01,Hot Dog,3,4.50
4,V-18,Rain Poncho,5,6.00
5,V-10,UVA T-Shirt,1,24.00
6,V-05,Chicken Tacos,4,6.50
7,V-18,Rain Poncho,8,6.00
8,V-42,Kettle Corn,3,5.00'''))

vendors = pd.read_csv(StringIO('''vendor_id,vendor_name,zone
V-01,Hoos Burgers,A
V-05,Rotunda Tacos,B
V-10,Cav Merch North,A
V-18,Rally Rain Gear,C
V-18,Rally Rain Gear,C'''))

targets = pd.read_csv(StringIO('''zone,revenue_target
A,80
B,25
C,60'''))

print('orders:', orders.shape, '| vendors:', vendors.shape, '| targets:', targets.shape)
orders

orders: (8, 5) | vendors: (5, 3) | targets: (3, 2)


,order_id,vendor_id,item,qty,price
0,1,V-01,Cheeseburger,2,7.5
1,2,V-10,Foam Finger,1,12.0
2,3,V-01,Hot Dog,3,4.5
3,4,V-18,Rain Poncho,5,6.0
4,5,V-10,UVA T-Shirt,1,24.0
5,6,V-05,Chicken Tacos,4,6.5
6,7,V-18,Rain Poncho,8,6.0
7,8,V-42,Kettle Corn,3,5.0


### Worked example — the merge, done carefully

Here is one merge done properly, so the pattern is on the screen before you write anything. Three things happen: record the baseline, merge with `indicator=True`, then compare against the baseline.

In [8]:
baseline_rows = len(orders)
orders['revenue'] = orders['qty'] * orders['price']
baseline_revenue = orders['revenue'].sum()
print(f'before: {baseline_rows} rows, ${baseline_revenue:.2f}')

check = orders.merge(vendors, on='vendor_id', how='left', indicator=True)
print(f'after:  {len(check)} rows, ${check["revenue"].sum():.2f}')
print()
print(check['_merge'].value_counts())

before: 8 rows, $183.50
after:  10 rows, $261.50

_merge
both          9
left_only     1
right_only    0
Name: count, dtype: int64


Eight orders went in and **ten** came out, and the revenue total jumped by $78. Both symptoms point at the same cause, and it is in the `vendors` table, not in the orders.

Note that `_merge` says `both = 9` and `left_only = 1`. Nine matches out of eight orders is already impossible, which is the tell.

Find it before you go further — everything downstream inherits this bug.

In [9]:
# Which vendor_id appears more than once in the vendor list?
print(vendors['vendor_id'].value_counts())
print()
print('duplicated vendor rows:', vendors.duplicated().sum())

vendor_id
V-18    2
V-01    1
V-05    1
V-10    1
Name: count, dtype: int64

duplicated vendor rows: 1


### Build 1 — fix the vendor list, then merge

**TODO:** drop the duplicate vendor row, then join it onto `orders` with an indicator. Your merge must come out at **8 rows** with the revenue total unchanged from the baseline. Print both to prove it.

In [10]:
# TODO: clean_vendors = ...
clean_vendors = vendors.drop_duplicates()
# TODO: joined = orders.merge(...)
joined = orders.merge(clean_vendors, on="vendor_id", how='left', indicator=True)
# TODO: print row count and revenue, and compare to baseline_rows / baseline_revenue
print(len(joined))
print(joined['revenue'].sum())
print(joined['revenue'].sum() == baseline_revenue)

8
183.5
True


### Build 2 — handle the vendor nobody has heard of

One order belongs to a vendor that is not on the roster. You have three options, and this is a judgment call:

1. Drop it — clean report, understated revenue.
2. Keep it with a blank name — the revenue total stays right, but it shows up as `NaN` in every chart and table.
3. Label it `'Unknown vendor'` and keep it in a zone called `'Unassigned'`.

**TODO:** pick one, implement it, and write one sentence saying why. Print how much revenue the decision affects either way.

In [11]:
# TODO: implement your choice
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')
joined['zone'] = joined['zone'].fillna('Unassigned')
# TODO: print the revenue attached to the unmatched order
print(joined.loc[joined['vendor_name'] == 'Unknown vendor', 'revenue'].sum())

15.0


**My decision, and why:** I picked option 3, because this way it keeps the revenue accurate and you're able to state that the vendor isn't on the roster.

### Build 3 — the per-vendor summary

**TODO:** one row per vendor, with:

- `orders` — how many orders
- `units` — total quantity
- `revenue` — total revenue
- `avg_ticket` — average revenue per order, rounded to 2 decimals

Sorted by revenue, highest first. Use `.agg()` with named outputs so the columns come out with the names above.

In [12]:
# TODO
joined.groupby('vendor_id').agg(
    orders=('order_id', 'count'),
    units=('qty', 'sum'),
    revenue=('revenue', 'sum'),
    avg_ticket=('revenue', 'mean'),
).round(2).sort_values('revenue', ascending=False)

,orders,units,revenue,avg_ticket
vendor_id,,,,
V-18,2,13,78.0,39.00
V-10,2,2,36.0,18.00
V-01,2,5,28.5,14.25
V-05,1,4,26.0,26.00
V-42,1,3,15.0,15.00


### Build 4 — did each zone hit its target?

**TODO:** total revenue by zone, join `targets` on, and add a `hit_target` boolean column. Then print a one-line sentence for each zone that a manager could read.

In [13]:
# TODO
rev_by_zone = joined.groupby('zone').agg(
    revenue= ('revenue', 'sum')
)
rev_by_zone = rev_by_zone.merge(targets, on='zone', how='left')
rev_by_zone['hit_target'] = rev_by_zone['revenue'] >= rev_by_zone['revenue_target']

for i, row in rev_by_zone.iterrows():
  print(f"Zone {row['zone']} hit target: {row['hit_target']}")

Zone A hit target: False
Zone B hit target: True
Zone C hit target: True
Zone Unassigned hit target: False


### Build 5 — the one number that matters *(stretch, if you have time)*

**TODO:** rain gear is the thing we can actually act on. Print total poncho units sold and what share of overall revenue they represent, as a percentage rounded to one decimal.

In [14]:
# TODO

poncho_units = joined.loc[joined['item'] == 'Rain Poncho', 'qty'].sum()
poncho_revenue = joined.loc[joined['item'] == 'Rain Poncho', 'revenue'].sum()
share = poncho_revenue / joined['revenue'].sum() * 100

print('Poncho units: ', poncho_units)
print(f"Revenue share: {share:.1f}%")

Poncho units:  13
Revenue share: 42.5%


---

## Checkpoint (participation)

Report the one line that fixes the copy trap, your row count and revenue total after the merge, and what you decided to do with the unknown vendor.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [15]:
# Checkpoint
copy_fix = sales.loc[sales['item'] == 'Hot Dog', 'price'] = 5.00         # TODO: the line you wrote to change `sales` itself
rows_after_merge = 8    # TODO: should equal 8
revenue_after_merge = 183.5 # TODO: should equal the baseline
unknown_vendor_call = ' picked option 3, because this way it keeps the revenue accurate and you are able to state that the vendor is not on the roster.'  # TODO: what you did with V-42, and why
top_vendor = 'V-18'        # TODO: highest-revenue vendor from your Build 3 summary

print('copy fix:', copy_fix)
print('rows after merge:', rows_after_merge)
print('revenue after merge:', revenue_after_merge)
print('unknown vendor:', unknown_vendor_call)
print('top vendor:', top_vendor)

copy fix: 5.0
rows after merge: 8
revenue after merge: 183.5
unknown vendor:  picked option 3, because this way it keeps the revenue accurate and you are able to state that the vendor is not on the roster.
top vendor: V-18
